# Tile rendering: per-scene gamma vs per-tile gamma

Compares the two renderings of the 100 annotation tiles:

| | endpoints (contrast) | gamma (exposure) |
|---|---|---|
| **before** | per scene | **per scene** |
| **after** | per scene | **per tile** |

Endpoints are per scene in both, so neither amplifies a flat tile into noise. Only
the tone curve changed.

**This notebook is self-contained.** Both renderings are pure functions of the
tile GeoTIFFs, so it recomputes them from the DN rather than depending on saved
PNGs. The `after` reconstruction is asserted identical to the PNGs on disk, which
makes the `before` reconstruction trustworthy by the same code path.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from PIL import Image

REPO = Path.cwd()
sys.path.insert(0, str(REPO / "train_scripts"))
from tile_stretch import (
    fit_endpoints, solve_gamma, to_uint8,
    TARGET_MEDIAN, GAMMA_MIN, GAMMA_MAX, LO_PCT, HI_PCT,
)

TILES = Path("/media/jvpcms/9a4d7913-803b-4816-9d70-c54550ae61b8/cbers_work/annotation_tiles")
manifest = pd.read_csv(TILES / "manifest.csv")
print(len(manifest), "tiles across", manifest.scene_id.nunique(), "scenes")
print("endpoints: p%.1f .. p%.1f   target median %.0f   gamma clamp [%.2f, %.2f]"
      % (LO_PCT, HI_PCT, TARGET_MEDIAN, GAMMA_MIN, GAMMA_MAX))

## Recompute both renderings from the DN GeoTIFFs

`before` solves one gamma per scene from the pooled median; `after` solves one per
tile. Everything else is identical.

In [ ]:
def scene_gamma(blocks, lo, hi):
    """The old rule: one exponent per scene, from the pooled median."""
    x = np.hstack([b[:3].reshape(3, -1) for b in blocks]).astype("f4")
    x = x[:, (x > 0).all(axis=0)]
    med = float(np.median(np.clip((x - lo[:, None]) / (hi - lo)[:, None], 0, 1)))
    med = min(max(med, 1e-3), 1 - 1e-6)
    return float(np.clip(np.log(TARGET_MEDIAN / 255.0) / np.log(med), GAMMA_MIN, GAMMA_MAX))

def tif_path(tile_id):
    return next(TILES.rglob(f"{tile_id}.tif"))

def png_path(tile_id):
    return next(TILES.rglob(f"{tile_id}.png"))

before, after, meta = {}, {}, {}
for scene, grp in manifest.groupby("scene_id"):
    ids = list(grp.tile_id)
    blocks = {t: rasterio.open(tif_path(t)).read() for t in ids}
    lo, hi = fit_endpoints(list(blocks.values()))
    g_scene = scene_gamma(list(blocks.values()), lo, hi)
    for t, b in blocks.items():
        g_tile = solve_gamma(b[:3].astype("f4"), lo, hi)
        before[t] = to_uint8(b[:3], lo, hi, g_scene).transpose(1, 2, 0)
        after[t]  = to_uint8(b[:3], lo, hi, g_tile ).transpose(1, 2, 0)
        meta[t] = dict(scene=scene, gamma_before=1 / g_scene, gamma_after=1 / g_tile,
                       lo=lo.copy(), hi=hi.copy())
print("recomputed", len(after), "tiles")

In [ ]:
# The `after` reconstruction must match what the cropper wrote, or the `before`
# reconstruction cannot be trusted either.
worst = max(int(np.abs(after[t].astype(int) - np.asarray(Image.open(png_path(t))).astype(int)).max())
            for t in after)
assert worst == 0, f"reconstruction differs from disk by {worst}"
print("verified: `after` is bit-identical to the PNGs on disk")

## Summary

In [ ]:
def stats(a):
    return float(np.median(a)), float((a >= 250).mean()), float((a <= 5).mean())

rows = []
for t in sorted(after):
    mb, cb, db = stats(before[t])
    ma_, ca, da = stats(after[t])
    rows.append(dict(tile=t, scene=meta[t]["scene"],
                     split=manifest.set_index("tile_id").loc[t, "split_use"],
                     med_before=mb, med_after=ma_,
                     clip_before=cb, clip_after=ca,
                     dark_before=db, dark_after=da,
                     gamma_before=meta[t]["gamma_before"], gamma_after=meta[t]["gamma_after"]))
df = pd.DataFrame(rows)
df["shift"] = (df.med_after - df.med_before).abs()

for label, col in (("rendered median", ("med_before", "med_after")),):
    b, a = df[col[0]], df[col[1]]
    print(f"{label}:  before {b.min():5.0f}..{b.max():<5.0f} std {b.std():5.1f}"
          f"   after {a.min():5.0f}..{a.max():<5.0f} std {a.std():5.1f}")
print(f"clipped >=250:   before mean {100*df.clip_before.mean():.3f}% worst {100*df.clip_before.max():.2f}%"
      f"   after mean {100*df.clip_after.mean():.3f}% worst {100*df.clip_after.max():.2f}%")
print(f"crushed <=5:     before mean {100*df.dark_before.mean():.3f}% worst {100*df.dark_before.max():.2f}%"
      f"   after mean {100*df.dark_after.mean():.3f}% worst {100*df.dark_after.max():.2f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df.med_before, bins=25, alpha=.65, label="before (scene gamma)")
ax[0].hist(df.med_after, bins=25, alpha=.65, label="after (tile gamma)")
ax[0].axvline(TARGET_MEDIAN, color="k", ls="--", lw=1, label=f"target {TARGET_MEDIAN:.0f}")
ax[0].set_xlabel("rendered median"); ax[0].set_ylabel("tiles"); ax[0].legend(fontsize=8)
ax[0].set_title("Exposure spread across 100 tiles")

ax[1].scatter(df.med_before, df.med_after, s=18)
ax[1].axhline(TARGET_MEDIAN, color="k", ls="--", lw=1)
ax[1].plot([0, 255], [0, 255], color="grey", lw=.8)
ax[1].set_xlabel("median before"); ax[1].set_ylabel("median after")
ax[1].set_title("Per-tile shift toward the target")
plt.tight_layout(); plt.show()

## Biggest changes first

In [ ]:
df.sort_values("shift", ascending=False).head(12)[
    ["tile", "med_before", "med_after", "clip_before", "clip_after", "gamma_after"]
]

In [ ]:
def compare(tile, downsample=2):
    b, a = before[tile], after[tile]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7.2))
    ax[0].imshow(b[::downsample, ::downsample])
    ax[1].imshow(a[::downsample, ::downsample])
    mb, cb, _ = stats(b); ma_, ca, _ = stats(a)
    ax[0].set_title(f"before   median {mb:.0f}   clipped {100*cb:.2f}%"
                    f"   gamma {meta[tile]['gamma_before']:.2f}")
    ax[1].set_title(f"after    median {ma_:.0f}   clipped {100*ca:.2f}%"
                    f"   gamma {meta[tile]['gamma_after']:.2f}")
    for x in ax: x.axis("off")
    split = manifest.set_index("tile_id").loc[tile, "split_use"]
    fig.suptitle(f"{tile}   [{split}]", fontsize=10)
    plt.tight_layout(); plt.show()

### The three tiles reported as too dark (207/153)

In [ ]:
for t in sorted(x for x in after if "207_153" in x and x.endswith(("29_46", "35_20", "39_39"))):
    compare(t)

### The tile reported as overexposed (212/151)

In [ ]:
for t in sorted(x for x in after if x.endswith("212_151_L4_42_18")):
    compare(t)

## All tiles, largest exposure change first

`LIMIT = None` for all 100.

In [ ]:
order = df.sort_values("shift", ascending=False).tile.tolist()
LIMIT = 20
for t in order[:LIMIT]:
    compare(t)

## Contact sheet

Each pair is before (left) then after (right).

In [ ]:
THUMB = 96
def thumb(a):
    step = max(1, a.shape[0] // THUMB)
    return a[::step, ::step][:THUMB, :THUMB]

cols = 8
n = len(order); rows_n = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows_n, cols * 2, figsize=(cols * 3.0, rows_n * 1.6))
axes = np.atleast_2d(axes)
for k, t in enumerate(order):
    r, c = divmod(k, cols)
    for off, src in ((0, before), (1, after)):
        ax = axes[r, c * 2 + off]
        ax.imshow(thumb(src[t])); ax.axis("off")
for k in range(n, rows_n * cols):
    r, c = divmod(k, cols)
    axes[r, c * 2].axis("off"); axes[r, c * 2 + 1].axis("off")
plt.tight_layout(); plt.show()

## Per-scene view

Scenes with a wide `spread_before` are the bimodal ones that motivated the change:
one exposure could not serve both their bright-water and land tiles.

In [ ]:
g = df.groupby("scene").agg(
    n=("tile", "size"),
    med_before=("med_before", "mean"), med_after=("med_after", "mean"),
    spread_before=("med_before", "std"), spread_after=("med_after", "std"),
    gamma_before=("gamma_before", "first"),
).sort_values("spread_before", ascending=False)
g.round(1)

## Where the gamma clamp binds

These tiles cannot reach the target: all-water tiles have no mid-tones to move,
so the clamp stops the curve going absurd. Expected, not a defect.

In [ ]:
edge = df[(df.gamma_after <= 1 / GAMMA_MAX + 1e-6) | (df.gamma_after >= 1 / GAMMA_MIN - 1e-6)]
edge[["tile", "med_after", "gamma_after"]]